# Active Learning Track A

Эксперимент сравнивает entropy, margin и random: один стратифицированный test, один initial set (`N=50`) и один pool, затем пять query-итераций по 20. Вся логика находится в production `ActiveLearningAgent`; notebook только запускает её по явному флагу или читает артефакты.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / 'config.yaml').exists() and (candidate / 'agents').exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from agents.al_agent import ActiveLearningAgent
from agents.common import load_frame, read_yaml

config = read_yaml(ROOT / 'config.yaml')
al_config = config['active_learning']
agent = ActiveLearningAgent(
    model=al_config['model'],
    config=al_config,
    random_seed=al_config['random_seed'],
)

## Методология без selection leakage

`source_label` пула является **симуляционным oracle**: query не использует её для выбора, а раскрывает только после selection. Это позволяет воспроизводимо сравнить стратегии, но не является утверждением о новой ручной разметке. Человеческая проверка проекта — отдельная review queue из AnnotationAgent.

In [ ]:
RUN_EXPERIMENT = False
final_candidates = [
    ROOT / 'data/labeled/reviews_final.parquet',
    ROOT / 'data/labeled/reviews_final.csv',
]
final_path = next((path for path in final_candidates if path.exists()), None)
results_path = ROOT / 'reports/active_learning/al_results.json'

if RUN_EXPERIMENT and final_path is not None:
    final_df = load_frame(final_path)
    results = agent.compare_strategies(
        final_df,
        strategies=al_config['strategies'],
        initial_size=al_config['initial_size'],
        n_iterations=al_config['n_iterations'],
        batch_size=al_config['batch_size'],
        test_size=al_config['test_size'],
        oracle_label_col='source_label',
    )
    agent.report(results, ROOT / 'reports/active_learning')
    print('Experiment completed with production agent.')
elif RUN_EXPERIMENT:
    results = None
    print('Final labeled dataset отсутствует. Сначала завершите проверенный HITL run.')
elif results_path.exists():
    results = json.loads(results_path.read_text(encoding='utf-8'))
    print(f'Loaded: {results_path}')
else:
    results = None
    print('AL artifacts отсутствуют. Завершите второй pipeline run либо включите RUN_EXPERIMENT.')

In [ ]:
if results is not None:
    histories = results['histories']
    for strategy, history in histories.items():
        print(f'\n{strategy}')
        display(pd.DataFrame(history)[['iteration', 'n_labeled', 'pool_remaining', 'accuracy', 'f1_macro']])
    manifest = results['split_manifest']
    display({
        'initial_size': len(manifest['initial_record_ids']),
        'pool_size': len(manifest['pool_record_ids']),
        'test_size': len(manifest['test_record_ids']),
        'random_seed': manifest['random_seed'],
        'oracle': manifest['oracle_label_col'],
    })
else:
    print('Нет результатов для отображения.')

In [ ]:
curve_path = ROOT / 'reports/active_learning/learning_curves.png'
report_path = ROOT / 'reports/active_learning/al_report.md'
if curve_path.exists():
    display(Image(filename=str(curve_path)))
else:
    print('Learning curve пока не создана.')
if report_path.exists():
    display(Markdown(report_path.read_text(encoding='utf-8')))

## Как формулировать вывод

Сравнивать нужно macro F1 при одинаковом числе раскрытых меток. Число «сэкономленных» примеров допустимо указывать только если entropy/margin действительно достигли финального F1 random baseline раньше; если порог не достигнут, корректный вывод — `not reached`, а не отрицательная или придуманная экономия. Дополнительно следует отметить variance одного seed и повторить эксперимент с несколькими seed в будущей работе.